In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from trace_comparison.triplet_accuracy import triplet_accuracy, relatedness_levels
from utils.notebook_utils import reset_cwd_and_env

reset_cwd_and_env()
base_path = os.path.join(os.getcwd(), f"data/similarity_matrix/nonograms")

In [ ]:
comparisons = ["cka", "cosim", "rmse", "clf_latent", "clf_weights", "imt_latent", "imt_weights", "temp"]
sim_matrices = {
    key: pd.read_csv(os.path.join(base_path, f"{key}.csv"), index_col=0) 
    for key in comparisons
}
tables = {
    key: triplet_accuracy(sim.to_numpy(), relatedness_levels(sim.index))
    for key, sim in sim_matrices.items()
}

In [ ]:
import pandas as pd

labels = ['ac_sgl_1', 'ac_sgl_2', 'ac_sgl_3', 'ac_pop_1', 'ac_pop_2', 'ac_pop_3',
          'ac_prl_1', 'bs_pop_1', 'bs_pop_2', 'bs_pop_3', 'bs_sgl_1', 'bs_sgl_2',
          'bs_sgl_3', 'gd_adm_1', 'gd_sgd_1', 'ga_rlt_1', 'ga_rlt_2', 'ga_rlt_3',
          'ga_tmt_1', 'ga_tmt_2', 'ga_tmt_3', 'nn_rml_1']

def sim(a, b):
    if a == b:
        return 1.0
    a1, a2, _ = a.split('_')
    b1, b2, _ = b.split('_')
    if a1 == b1 and a2 == b2:
        return 0.75
    if a1 == b1:
        return 0.5
    return 0.25

sim_matrix = pd.DataFrame(
    [[sim(a, b) for b in labels] for a in labels],
    index=labels, columns=labels
)

sim_matrices['ground_truth'] = sim_matrix
sim_matrices['cjask'] = sim_matrices['clf_latent'] + 3 *sim_matrices['temp']
sim_matrices['cjask'] /= 4

In [ ]:
# plot similarity matrices
for key, sim in sim_matrices.items():
    plt.figure(figsize=(8, 6))
    sns.heatmap(sim, cmap="viridis", square=True, cbar_kws={"shrink": 0.8})
    plt.title(f"{key.upper()} Similarity Matrix")
    plt.xlabel("Trace Index")
    plt.ylabel("Trace Index")
    plt.tight_layout()
    plt.show()

In [ ]:
for key, table in tables.items():
    print(f"============= {key.upper()} ==============")
    print(table)
    print()

In [ ]:
"""Triplet accuracy: random baseline (median ±1σ/±2σ) vs. real matrices."""

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LEVEL_PAIRS = [(2, 1), (2, 0), (1, 0)]


def relatedness_levels(names):
    parts = [n.split("_") for n in names]
    solver = np.array([p[0] for p in parts])
    variant = np.array(["_".join(p[:2]) for p in parts])
    return (solver[:, None] == solver).astype(int) + (variant[:, None] == variant)


def triplet_accuracy_fast(sim, level):
    """Vectorized over anchors — identical result to the loop version."""
    n = len(sim)
    valid = ~np.eye(n, dtype=bool)                # (n, n): off-diagonal
    hit = sim[:, :, None] > sim[:, None, :]       # (n, n, n): hit[a,p,q] = sim[a,p] > sim[a,q]
    counts = np.zeros((len(LEVEL_PAIRS), 2), dtype=np.int64)
    for k, (hi, lo) in enumerate(LEVEL_PAIRS):
        hi_mask = (level == hi) & valid           # (n, n)
        lo_mask = (level == lo) & valid
        pairs = hi_mask[:, :, None] & lo_mask[:, None, :]   # (n, n, n)
        counts[k, 0] = np.logical_and(pairs, hit).sum()
        counts[k, 1] = pairs.sum()
    table = pd.DataFrame(
        counts,
        columns=["correct", "total"],
        index=[f"L{hi} > L{lo}" for hi, lo in LEVEL_PAIRS],
    )
    table["accuracy"] = table["correct"] / table["total"]
    return table


def random_sim_matrix(labels, rng, low=0.0, high=1.0):
    """Symmetric, unit diagonal, values uniform in [low, high)."""
    n = len(labels)
    A = rng.uniform(low, high, size=(n, n))
    A = (A + A.T) / 2
    np.fill_diagonal(A, 1.0)
    return pd.DataFrame(A, index=labels, columns=labels)


def run_trials(labels, n_trials=1000, seed=0, gen=random_sim_matrix):
    rng = np.random.default_rng(seed)
    level = relatedness_levels(labels)
    rows = []
    for _ in range(n_trials):
        sim = gen(labels, rng)
        table = triplet_accuracy_fast(sim.to_numpy(), level)
        row = table["accuracy"].to_dict()
        row["Overall"] = table["correct"].sum() / table["total"].sum()
        rows.append(row)
    return pd.DataFrame(rows)


def summarize(trials):
    return pd.DataFrame({
        "median": trials.median(),
        "mean":   trials.mean(),
        "std":    trials.std(),           # ddof=1
        "q2.5":   trials.quantile(0.025),
        "q16":    trials.quantile(0.16),
        "q84":    trials.quantile(0.84),
        "q97.5":  trials.quantile(0.975),
    })


def plot_bands(trials, real_accuracies, out_path=None):
    cats = list(trials.columns)
    x = np.arange(len(cats))
    med = trials.median().to_numpy()
    std = trials.std().to_numpy()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.fill_between(x, med - 2 * std, med + 2 * std, alpha=0.12, color="C0", label="±2σ")
    ax.fill_between(x, med - std,     med + std,     alpha=0.25, color="C0", label="±1σ")
    ax.plot(x, med, color="C0", marker="o", lw=2, label="median")
    ax.plot(x, med + std,     color="C0", ls="--", lw=1, alpha=0.7)
    ax.plot(x, med - std,     color="C0", ls="--", lw=1, alpha=0.7)
    ax.plot(x, med + 2 * std, color="C0", ls=":",  lw=1, alpha=0.7)
    ax.plot(x, med - 2 * std, color="C0", ls=":",  lw=1, alpha=0.7)

    for i, (name, acc) in enumerate(real_accuracies.items()):
        vals = [acc[c] for c in cats]
        ax.plot(x, vals, marker="s", lw=1.5, label=name, color=f"C{i + 1}")
        for xi, v in zip(x, vals):
            ax.annotate(f"{v:.3f}", (xi, v), textcoords="offset points",
                        xytext=(0, 6), ha="center", fontsize=7, color=f"C{i + 1}")

    ax.set_xticks(x)
    ax.set_xticklabels(cats)
    ax.set_ylabel("Triplet accuracy")
    ax.set_title(f"Triplet accuracy: random baseline (N={len(trials)}) vs. real matrices")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=150)
    return fig, ax


def main():
    base_path = Path(os.environ.get("DATA_DIR", "data")) / "similarity_matrix" / "nonograms"
    sim_matrices = {k: pd.read_csv(base_path / f"{k}.csv", index_col=0) for k in comparisons}

    labels = sim_matrices["cka"].index
    for k, sim in sim_matrices.items():
        assert sim.index.equals(labels) and sim.columns.equals(labels), k

    # 1) null distribution from random matrices
    trials = run_trials(labels, n_trials=1000, seed=0)

    # 2) real accuracies
    level = relatedness_levels(labels)
    real = {}
    for k, sim in sim_matrices.items():
        table = triplet_accuracy_fast(sim.to_numpy(), level)
        acc = table["accuracy"].to_dict()
        acc["Overall"] = table["correct"].sum() / table["total"].sum()
        real[k] = acc

    print("Random baseline (N=1000):")
    print(summarize(trials).round(4))
    print("\nReal accuracies:")
    print(pd.DataFrame(real).round(4))
    print("\nEmpirical p = fraction of random trials with accuracy >= real:")
    for k, acc in real.items():
        print(f"  {k}: " + ", ".join(
            f"{c}={float((trials[c] >= acc[c]).mean()):.3f}" for c in trials.columns
        ))

    plot_bands(trials, real, out_path="triplet_accuracy_baseline.png")
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
"""Triplet accuracy: random-matrix null (median, ±1σ/±2σ/±3σ) vs. real matrices.

A triplet is (anchor a, p, q) with level(a, p) = hi > lo = level(a, q).
It is "correct" when the matrix says a is strictly more similar to p than to q.
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from math import erf, sqrt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LEVEL_PAIRS = [(2, 1), (2, 0), (1, 0)]
PAIR_NAMES = [f"L{hi} > L{lo}" for hi, lo in LEVEL_PAIRS]
CATEGORIES = PAIR_NAMES + ["Overall"]
SIGMAS = (1, 2, 3)

# True  -> matrix is used exactly as stored (larger value = more similar).
# False -> matrix is a distance/error and is negated before scoring.
# All True reproduces the original script. Flip an entry only if that file
# really holds a distance (lower = more similar).
HIGHER_IS_MORE_SIMILAR = {"cka": True, "cosim": True, "rmse": True, "clf_emb": True}


# --------------------------------------------------------------------------- #
# Triplet accuracy
# --------------------------------------------------------------------------- #
def relatedness_levels(names) -> np.ndarray:
    """2 = same solver and variant, 1 = same solver only, 0 = unrelated."""
    parts = [str(n).split("_") for n in names]
    solver = np.array([p[0] for p in parts])
    variant = np.array(["_".join(p[:2]) for p in parts])
    return (solver[:, None] == solver).astype(int) + (variant[:, None] == variant)


@dataclass(frozen=True)
class TripletIndex:
    """Anchor-wise index sets per level. Depends only on the labels, so it is
    built once and reused across all random trials."""
    groups: list          # groups[a][level] -> indices j != a with level[a, j] == level
    totals: np.ndarray    # (len(LEVEL_PAIRS),) number of valid triplets per pair

    @classmethod
    def from_levels(cls, level: np.ndarray) -> "TripletIndex":
        n = len(level)
        off_diag = ~np.eye(n, dtype=bool)
        groups = [
            [np.flatnonzero((level[a] == l) & off_diag[a]) for l in (0, 1, 2)]
            for a in range(n)
        ]
        sizes = np.array([[len(g) for g in anchor] for anchor in groups])  # (n, 3)
        totals = np.array([(sizes[:, hi] * sizes[:, lo]).sum() for hi, lo in LEVEL_PAIRS])
        return cls(groups, totals)


def triplet_accuracy(sim: np.ndarray, index: TripletIndex) -> pd.Series:
    """Accuracy per level pair plus 'Overall'. Ties count as incorrect.

    For each anchor, the number of (p, q) pairs with sim[a, p] > sim[a, q] is found
    by sorting the q-values and binary-searching each p-value. That is
    O(n² log n) time and O(n) memory, versus O(n³) for the broadcast version.
    """
    correct = np.zeros(len(LEVEL_PAIRS), dtype=np.int64)
    for a, idx in enumerate(index.groups):
        vals = [sim[a, i] for i in idx]
        srt = [np.sort(v) for v in vals]
        for k, (hi, lo) in enumerate(LEVEL_PAIRS):
            # searchsorted(..., 'left') = number of lo-values strictly below each hi-value
            correct[k] += np.searchsorted(srt[lo], vals[hi], side="left").sum()
    with np.errstate(invalid="ignore", divide="ignore"):
        acc = correct / index.totals
        overall = correct.sum() / index.totals.sum()
    return pd.Series([*acc, overall], index=CATEGORIES)


# --------------------------------------------------------------------------- #
# Null distribution
# --------------------------------------------------------------------------- #
def random_sim_matrix(n: int, rng: np.random.Generator) -> np.ndarray:
    """Symmetric matrix with i.i.d. U[0, 1) off-diagonal entries.
    The diagonal is never read by the scorer, so it is left at 0."""
    upper = np.triu(rng.random((n, n)), k=1)
    return upper + upper.T


def run_trials(index: TripletIndex, n: int, n_trials: int = 2000, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = [triplet_accuracy(random_sim_matrix(n, rng), index) for _ in range(n_trials)]
    return pd.DataFrame(rows).reset_index(drop=True)


def normal_cdf(z: float) -> float:
    return 0.5 * (1.0 + erf(z / sqrt(2.0)))


def summarize(trials: pd.DataFrame) -> pd.DataFrame:
    out = {"median": trials.median(), "mean": trials.mean(), "std": trials.std()}  # ddof=1
    for k in SIGMAS:
        out[f"q(-{k}σ)"] = trials.quantile(normal_cdf(-k))
        out[f"q(+{k}σ)"] = trials.quantile(normal_cdf(k))
    return pd.DataFrame(out)


def compare_to_null(real: pd.DataFrame, trials: pd.DataFrame):
    """z-scores and one-sided empirical p-values (P[null >= real]).

    p uses the (1 + count) / (N + 1) correction, so it is never exactly 0.
    """
    n = len(trials)
    z = real.sub(trials.mean(), axis=0).div(trials.std(), axis=0)
    p = pd.DataFrame(
        {
            name: [(1 + (trials[c] >= real.loc[c, name]).sum()) / (n + 1) for c in real.index]
            for name in real.columns
        },
        index=real.index,
    )
    return z, p


# --------------------------------------------------------------------------- #
# Plot
# --------------------------------------------------------------------------- #
def plot_bands(trials: pd.DataFrame, real: pd.DataFrame, out_path: str | None = None):
    x = np.arange(len(CATEGORIES))
    med = trials.median()[CATEGORIES].to_numpy()
    std = trials.std()[CATEGORIES].to_numpy()
    half = 0.35

    fig, ax = plt.subplots(figsize=(10, 5.5))
    # Widest band first; the translucent bands stack, so the core looks darker.
    for k in sorted(SIGMAS, reverse=True):
        ax.bar(x, 2 * k * std, bottom=med - k * std, width=2 * half,
               color="C0", alpha=0.15, label=f"±{k}σ", zorder=1)
    ax.hlines(med, x - half, x + half, color="C0", lw=2.5, label="random median", zorder=2)

    offsets = np.linspace(-0.2, 0.2, len(real.columns)) if len(real.columns) > 1 else [0.0]
    for i, (name, off) in enumerate(zip(real.columns, offsets)):
        vals = real[name].reindex(CATEGORIES).to_numpy()
        color = f"C{i + 1}"
        ax.scatter(x + off, vals, marker="D", s=45, color=color,
                   edgecolor="k", linewidth=0.6, zorder=3, label=name)
        for xi, v in zip(x + off, vals):
            ax.annotate(f"{v:.3f}", (xi, v), textcoords="offset points",
                        xytext=(0, 8), ha="center", fontsize=7, color=color)

    lo = min((med - max(SIGMAS) * std).min(), np.nanmin(real.to_numpy()))
    hi = max((med + max(SIGMAS) * std).max(), np.nanmax(real.to_numpy()))
    ax.set_ylim(lo - 0.03, min(1.0, hi + 0.05) if hi < 1.0 else 1.05)

    ax.set_xticks(x)
    ax.set_xticklabels(CATEGORIES)
    ax.set_ylabel("Triplet accuracy")
    ax.set_title(f"Triplet accuracy: random baseline (N={len(trials)}) vs. real matrices")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=150)
    return fig, ax


# --------------------------------------------------------------------------- #
# I/O
# --------------------------------------------------------------------------- #
def load_matrices(base_path: Path, names: list[str]):
    """Load CSVs, align them to a common label order, and orient to 'higher = more similar'."""
    raw = {k: pd.read_csv(base_path / f"{k}.csv", index_col=0) for k in names}
    labels = raw[names[0]].index

    mats = {}
    for k, df in raw.items():
        if set(df.index) != set(labels) or set(df.columns) != set(labels):
            raise ValueError(f"{k}: row/column labels do not match those of '{names[0]}'")
        if df.index.has_duplicates:
            raise ValueError(f"{k}: duplicate labels")
        arr = df.reindex(index=labels, columns=labels).to_numpy(dtype=float)
        if np.isnan(arr).any():
            raise ValueError(f"{k}: contains NaNs")
        if not np.allclose(arr, arr.T):
            print(f"warning: {k} is not symmetric")
        mats[k] = arr if HIGHER_IS_MORE_SIMILAR.get(k, True) else -arr
    return labels, mats


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #
def main(
    data_dir: str | os.PathLike = os.environ.get("DATA_DIR", "data"),
    n_trials: int = 2000,      # use >= 10000 if you rely on the ±3σ quantiles
    seed: int = 0,
    out_path: str | None = "triplet_accuracy_baseline.png",
    show: bool = True,
):
    base_path = Path(data_dir) / "similarity_matrix" / "nonograms"
    labels, mats = load_matrices(base_path, list(HIGHER_IS_MORE_SIMILAR))

    level = relatedness_levels(labels)
    index = TripletIndex.from_levels(level)

    trials = run_trials(index, n=len(labels), n_trials=n_trials, seed=seed)
    real = pd.DataFrame({k: triplet_accuracy(m, index) for k, m in mats.items()})
    z, p = compare_to_null(real, trials)
    summary = summarize(trials)

    flipped = [k for k in real.columns if (z[k] < -3).any()]
    if flipped:
        print(f"NOTE: {flipped} score well BELOW the random baseline. If these files hold "
              "distances (lower = more similar), set them to False in HIGHER_IS_MORE_SIMILAR.\n")

    pd.set_option("display.width", 160)
    print(f"Random baseline (N={n_trials}):")
    print(summary.round(4))
    print("\nReal accuracies:")
    print(real.round(4))
    print("\nz-scores vs. random mean:")
    print(z.round(2))
    print("\nEmpirical one-sided p (P[random >= real], +1 corrected):")
    print(p.round(4))

    fig, ax = plot_bands(trials, real, out_path=out_path)
    if show:
        plt.show()

    return {"trials": trials, "summary": summary, "real": real,
            "z": z, "p": p, "fig": fig, "labels": labels}

In [ ]:
main()